In [2]:
import pandas as pd
import joblib
from sklearn.metrics import confusion_matrix, precision_score

rf_model = joblib.load("models/rf_model.pkl")
X_test = pd.read_csv("data/X_test.csv", index_col=0)
y_test = pd.read_csv("data/y_test.csv", index_col=0).squeeze()
df_clean = pd.read_csv("data/df_clean.csv")

y_pred_rf = rf_model.predict(X_test)

In [4]:
print(X_test.index.max(), "vs", len(df_clean), "rows in df_clean")

test_info = df_clean.loc[X_test.index, ['race', 'score_text']]
print(test_info.head())

6165 vs 6172 rows in df_clean
                  race score_text
5155          Hispanic       High
2421  African-American        Low
2394             Other       High
4941             Asian        Low
1603  African-American        Low


In [5]:
test_info = df_clean.loc[X_test.index, ['race', 'score_text']].copy()
test_info['compas_pred'] = test_info['score_text'].isin(['Medium', 'High']).astype(int)
test_info.head()

,race,score_text,compas_pred
5155,Hispanic,High,1
2421,African-American,Low,0
2394,Other,High,1
4941,Asian,Low,0
1603,African-American,Low,0


In [6]:
def error_rates_by_group(y_true, y_pred, group_labels, groups_to_check):
    results = []
    for g in groups_to_check:
        mask = group_labels == g
        cm = confusion_matrix(y_true[mask], y_pred[mask], labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else float('nan')
        fnr = fn / (fn + tp) if (fn + tp) > 0 else float('nan')
        results.append({'race': g, 'n': mask.sum(), 'FPR': fpr, 'FNR': fnr})
    return pd.DataFrame(results)

In [7]:
groups_to_check = ['African-American', 'Caucasian']

In [8]:
compas_results = error_rates_by_group(
    y_test, test_info['compas_pred'], test_info['race'], groups_to_check
)
print("COMPAS:")
print(compas_results)

COMPAS:
               race    n       FPR       FNR
0  African-American  641  0.401316  0.299703
1         Caucasian  427  0.206767  0.503106


In [9]:
y_pred_rf_series = pd.Series(y_pred_rf, index=X_test.index)

model_results = error_rates_by_group(
    y_test, y_pred_rf_series, test_info['race'], groups_to_check
)
print("Your Random Forest:")
print(model_results)

Your Random Forest:
               race    n       FPR       FNR
0  African-American  641  0.351974  0.228487
1         Caucasian  427  0.101504  0.645963


In [10]:
from sklearn.metrics import precision_score

for race in groups_to_check:
    mask = test_info['race'] == race
    prec = precision_score(y_test[mask], y_pred_rf_series[mask])
    print(f"{race} — Random Forest precision (of predicted reoffenders, % correct): {prec:.3f}")

African-American — Random Forest precision (of predicted reoffenders, % correct): 0.708
Caucasian — Random Forest precision (of predicted reoffenders, % correct): 0.679
